# 01 · Setup & Authentication

> **Module 2 · SDK Setup & The Runner · Lecture 2.1**

In this notebook you will:

- Verify that Apache Beam is installed and importable, and record the exact version you have
- Print a machine-readable description of your environment
- Understand the difference between the Direct Runner and the Dataflow Runner
- Learn the three ways to authenticate to Dataflow, and why none of them are needed locally
- Build and run your first Beam pipeline, then read the job graph it produced
- Reproduce a real Beam failure — a duplicate transform label — and read its actual error message

---

## 1. Verify the SDK

Everything in this course runs on the Apache Beam Python SDK. Before anything else, confirm it is installed and record the exact version, because the version determines which transforms and runner behaviours are available.

In [1]:
import sys
from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
print("course root:", COURSE_ROOT)

import apache_beam as beam

print(f"apache-beam version : {beam.__version__}")
print(f"python version      : {sys.version.split()[0]}")
print(f"beam module path    : {Path(beam.__file__).parent}")
print()
print("Core transforms available in this SDK:")
for name in ("Map", "FlatMap", "Filter", "Flatten", "ParDo", "CombinePerKey",
             "GroupByKey", "CoGroupByKey", "Create", "Reshuffle"):
    print(f"  beam.{name:<16} {hasattr(beam, name)}")


course root: /Users/pvishnoi/PycharmProjects/data-etl-ml/data-engineering/data-Course/google/Dataflow


apache-beam version : 2.77.0
python version      : 3.13.16
beam module path    : /Users/pvishnoi/PycharmProjects/data-etl-ml/data-engineering/data-Course/google/Dataflow/.venv/lib/python3.13/site-packages/apache_beam

Core transforms available in this SDK:
  beam.Map              True
  beam.FlatMap          True
  beam.Filter           True
  beam.Flatten          True
  beam.ParDo            True
  beam.CombinePerKey    True
  beam.GroupByKey       True
  beam.CoGroupByKey     True
  beam.Create           True
  beam.Reshuffle        True


### Why the version matters more than it looks

Beam's public API is stable, but its *runner* behaviour is not. Three concrete examples from this course:

| SDK version | What changed | Why it matters to you |
|---|---|---|
| 2.20.0 | `DataflowRunner` gained `--use_runner_v2` by default | Job graph shape changes; the Dataflow UI shows different stages |
| 2.60.0 | The Go-based "Prism" runner appeared under the plain name `DirectRunner` | Faster locally, but it panics on some trigger/window combinations |
| 2.77.0 | Current course baseline | `BundleBasedDirectRunner` is the safe local default |

The course pins its runner deliberately. `beam_options.py` sets:

```python
DEFAULT_RUNNER = os.environ.get("BEAM_RUNNER", "BundleBasedDirectRunner")
```

The name is confusing but the reason is concrete: Beam 2.60+ also exposes a Go runner under the plain name `DirectRunner` (the "Prism" runner). It is faster, and it is genuinely useful for large batch pipelines. But it does not yet support every trigger and windowing combination — it panics on the `AfterWatermark` trigger that the streaming chapter uses. The course therefore defaults to the classic pure-Python runner, and you can still opt into Prism explicitly with `--runner DirectRunner`.

## 2. Describe the environment

The project ships a `beam_options.describe_environment()` helper. It is deliberately cheap to call and safe to run with no cloud account: it reads only local state and environment variables. This is exactly the function you would call at the top of a support ticket.

In [2]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd() / "project"
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd().parent / "project"
sys.path.insert(0, str(PROJECT_DIR))

from beam_options import describe_environment

env = describe_environment()
width = max(len(k) for k in env)
print("Environment")
print("-" * (width + 40))
for key, value in env.items():
    print(f"  {key:<{width}} : {value}")


Environment
--------------------------------------------------------
  python_version   : 3.13.16
  beam_version     : 2.77.0
  project          : (not set)
  region           : us-central1
  temp_location    : (not set)
  staging_location : (not set)
  service_account  : (not set)


Note what `describe_environment()` reports: `project`, `temp_location`, `staging_location` and `service_account` all read `(not set)` when the corresponding environment variables are absent. That is the correct state for local work, and it is why every cell in this course that touches GCP is gated behind an explicit check.

## 3. The two runners

This is the single most important operational fact in the course:

| Runner | Where it runs | Credentials needed | Cost |
|---|---|---|---|
| `BundleBasedDirectRunner` (course default) | In this Python process | None | None |
| `DirectRunner` (Prism, Go-based) | In this Python process, native Go | None | None |
| `DataflowRunner` | Google Cloud, on Google-managed workers | Yes | Yes — vCPU-hours, and for streaming, forever until cancelled |

`is_local()` encapsulates the question "do I need credentials for this?" and every course command relies on it. The default is local, and asking for the cloud is an explicit opt-in.

One implementation detail worth knowing, because it will bite you if you read `beam_options.py` closely: `resolve_runner(options)` reads `options.runner`, and Beam only populates that attribute once a pipeline has actually been constructed. **Before** that, `options.runner` is `None` and `resolve_runner` falls back to `DEFAULT_RUNNER`. The reliable way to see which runner you asked for is `options.get_all_options()["runner"]`, which holds the raw flag string. The cell below uses that.

In [3]:
# How a --runner flag becomes an actual runner object.
from apache_beam.runners import runner as beam_runner

from beam_options import build_options, LOCAL_RUNNERS

scenarios = [
    ("no flags at all", []),
    ("explicit local runner", ["--runner", "BundleBasedDirectRunner"]),
    ("explicit Go runner", ["--runner", "DirectRunner"]),
    ("explicit cloud runner", ["--runner", "DataflowRunner"]),
]

print(f"{'scenario':<22} {'--runner flag':<26} {'runner class':<28} {'needs GCP?'}")
print("-" * 90)
for label, argv in scenarios:
    options = build_options(argv)
    flag = options.get_all_options().get("runner")
    runner_class = type(beam_runner.create_runner(flag)).__name__
    needs_gcp = "YES" if runner_class not in LOCAL_RUNNERS else "no"
    print(f"{label:<22} {flag:<26} {runner_class:<28} {needs_gcp}")

print()
print("Two things worth noticing in that table.")
print()
print("1. Beam wraps the Go runner in 'SwitchingDirectRunner', which dispatches")
print("   to the Go implementation for supported pipelines and falls back to the")
print("   Python one otherwise. That fallback is why the course still defaults")
print("   to BundleBasedDirectRunner for the streaming chapters.")
print()
print("2. 'SwitchingDirectRunner' is not itself in the course's LOCAL_RUNNERS")
print("   set, so a name-based 'is this local?' check must look at the flag you")
print("   asked for, not at the wrapper class Beam hands back.")


scenario               --runner flag              runner class                 needs GCP?
------------------------------------------------------------------------------------------
no flags at all        BundleBasedDirectRunner    BundleBasedDirectRunner      no
explicit local runner  BundleBasedDirectRunner    BundleBasedDirectRunner      no
explicit Go runner     DirectRunner               SwitchingDirectRunner        YES
explicit cloud runner  DataflowRunner             DataflowRunner               YES

Two things worth noticing in that table.

1. Beam wraps the Go runner in 'SwitchingDirectRunner', which dispatches
   to the Go implementation for supported pipelines and falls back to the
   Python one otherwise. That fallback is why the course still defaults
   to BundleBasedDirectRunner for the streaming chapters.

2. 'SwitchingDirectRunner' is not itself in the course's LOCAL_RUNNERS
   set, so a name-based 'is this local?' check must look at the flag you
   asked for, not at the

### The one thing that surprises people

`build_options([])` does **not** raise, and it does not ask for credentials. It silently produces a local runner. That is deliberate — the course, and most of the Beam ecosystem, wants local iteration to be the path of least resistance. The corollary is that a pipeline can be "green" locally and completely broken in the cloud, because the cloud runner adds requirements the local runner never checked.

Here is the real error you get when you skip them. This is the verbatim Beam 2.77 message:

In [4]:
# A deliberate failure: ask for the cloud runner without the cloud options.
from apache_beam.options.pipeline_options import PipelineOptions
import apache_beam as beam

try:
    with beam.Pipeline(options=PipelineOptions(["--runner", "DataflowRunner"])):
        pass
except Exception as exc:
    print("Exception type:", type(exc).__name__)
    print()
    print(exc)


Exception type: ValueError

Pipeline has validations errors: 
Missing GCS path option: temp_location.
Missing GCS path option: staging_location.
Missing required option: project.
Missing required option: region.


Read that error carefully — it names **all four** missing requirements at once, and it names them in the order Beam checks them:

1. `temp_location` — where Dataflow puts shuffle and spill data
2. `staging_location` — where Dataflow stages your worker code
3. `project` — which project to bill
4. `region` — where the workers run

**None of these are exercised by the local runner.** That is the single biggest reason a pipeline works on Monday and fails on Tuesday when someone adds `--runner DataflowRunner`. The local path checks nothing about your cloud setup, so a green local run tells you nothing about a green cloud run.

`beam_options.apply_dataflow_defaults(options, bucket=...)` exists precisely to close this gap: given a bucket, it derives both locations from bucket-specific prefixes, which also stops two concurrent jobs in the same region from colliding on the same staging area.

## 4. Authentication — explained, never required

You do **not** need a GCP account for anything in this course. But you will eventually need one, so here are the three options, ranked.

| Method | Credential lifetime | Where it belongs | Use when |
|---|---|---|---|
| **Application Default Credentials (ADC)** | ~1 hour, auto-refreshed | Developer machine, CI | Default for humans and CI |
| **Service account key file** | Until manually revoked | Secret manager, never git | Legacy systems that cannot do ADC |
| **Workload Identity Federation** | ~1 hour, auto-refreshed | GKE, Cloud Run, other clouds | Best option in any modern deployment |

The modern answer is **Workload Identity Federation** (or the equivalent attached service accounts on GKE/Cloud Run). Key files are a legacy compatibility path: they are long-lived secrets that leak from CI logs and git history.

For a developer laptop, ADC is one command:

```bash
gcloud auth application-default login
```

For a CI system, use a federated identity pool. The `gcloud` commands below are shown for reference only — **the notebook does not execute them, and they require no credentials for the cell to succeed.**

```bash
# Reference only — requires a real GCP project.
gcloud auth application-default login --impersonate-service-account=dataflow-runner@YOUR_PROJECT_ID.iam.gserviceaccount.com
gcloud projects add-iam-policy-binding YOUR_PROJECT_ID \
  --member="serviceAccount:dataflow-runner@YOUR_PROJECT_ID.iam.gserviceaccount.com" \
  --role="roles/dataflow.worker"
```

The cell below inspects what credentials, if any, are visible — without contacting Google.

In [5]:
import os
import shutil

print("Credential-related environment variables")
print("-" * 60)
for var in (
    "GOOGLE_APPLICATION_CREDENTIALS",
    "GOOGLE_CLOUD_PROJECT",
    "PROJECT_ID",
    "STAGING_LOCATION",
    "TEMP_LOCATION",
    "SERVICE_ACCOUNT_EMAIL",
):
    value = os.environ.get(var)
    print(f"  {var:<30} {value if value else '(not set)'}")

print()
gcloud = shutil.which("gcloud")
print("gcloud on PATH:", gcloud if gcloud else "no — the CLI is not installed")

# This is the gate every GCP cell in this course uses. It is a plain
# environment check: no credential discovery, no network call, no failure.
PROJECT_ID = os.environ.get("PROJECT_ID", "")
print()
print("PROJECT_ID gate:", PROJECT_ID if PROJECT_ID else "(not set — cloud cells will be skipped)")


Credential-related environment variables
------------------------------------------------------------
  GOOGLE_APPLICATION_CREDENTIALS (not set)
  GOOGLE_CLOUD_PROJECT           (not set)
  PROJECT_ID                     (not set)
  STAGING_LOCATION               (not set)
  TEMP_LOCATION                  (not set)
  SERVICE_ACCOUNT_EMAIL          (not set)

gcloud on PATH: no — the CLI is not installed

PROJECT_ID gate: (not set — cloud cells will be skipped)


### The gate pattern

Every cell that would touch Google Cloud in this course looks like this:

```python
if PROJECT_ID:
    # do the cloud thing
else:
    print("Skipped — set PROJECT_ID to enable this cell.")
```

The point is that the notebook is **always runnable**. A reader without a billing-enabled project still gets the full lesson; they simply get the skip message instead of a credential error. This matters more than it sounds: a course notebook that fails on machine 12 of 30 is a course that gets abandoned.

## 5. Your first pipeline

Now the actual work. The smallest complete Beam pipeline has three parts: a source, a transform, and a sink. Beam's `Create` is a synthetic source, and `beam.Map(print)` is the simplest possible sink.

In [6]:
import apache_beam as beam

with beam.Pipeline() as p:
    (
        p
        | "Create" >> beam.Create([1, 2, 3, 4, 5])
        | "Double" >> beam.Map(lambda x: x * 2)
        | "Print" >> beam.Map(print)
    )


2
4
6
8
10


### Why nothing printed when you built the PCollection

This is worth understanding rather than memorising. A PCollection is a **description**, not data. Building the graph does no work at all. The `with beam.Pipeline()` block runs the pipeline when it exits, and only then does the runner pull elements through the graph.

The consequence you will hit repeatedly: **a pipeline with no sink does nothing.** If you build a PCollection and never consume it, the runner may not even execute that branch. This is not a bug; it is how the model works. Every production pipeline ends each branch in a sink.

## 6. Read the job graph

Before you run a pipeline, you can inspect the graph it will execute. Beam renders the steps and their labels — this is the fastest way to find a mislabelled or misordered stage.

In [7]:
# Build the graph WITHOUT running it, so we can look at it first.
class GraphPrinter:
    """A minimal Beam graph visitor that prints each step's full label."""

    def enter_composite_transform(self, node):
        pass

    def leave_composite_transform(self, node):
        pass

    def visit_transform(self, node):
        print("   ", node.full_label)

    def visit_value(self, node, *args):
        pass


p = beam.Pipeline()
(
    p
    | "Create" >> beam.Create(["EMEA", "AMER", "APAC"])
    | "ToPairs" >> beam.Map(lambda region: (region, len(region)))
    | "SortByLength" >> beam.Map(lambda kv: (kv[1], kv[0]))
)
print("Pipeline graph:")
p.visit(GraphPrinter())
print()
print("Each line is one step. The quoted text is the label you gave.")
print("The runner is free to fuse these into fewer steps than you see here.")


Pipeline graph:
    [8]: Create/Impulse
    [8]: Create/FlatMap(<lambda at core.py:4094>)
    [8]: Create/MaybeReshuffle/Reshuffle/AddRandomKeys
    [8]: Create/MaybeReshuffle/Reshuffle/ReshufflePerKey/Map(reify_metadata_default_window)
    [8]: Create/MaybeReshuffle/Reshuffle/ReshufflePerKey/GroupByKey
    [8]: Create/MaybeReshuffle/Reshuffle/ReshufflePerKey/FlatMap(restore_metadata_default_window)
    [8]: Create/MaybeReshuffle/Reshuffle/RemoveRandomKeys
    [8]: Create/Map(decode)
    [8]: ToPairs
    [8]: SortByLength

Each line is one step. The quoted text is the label you gave.
The runner is free to fuse these into fewer steps than you see here.


### Read that graph carefully

Two things stand out, and both are useful.

**First, your labels are only part of the name.** `Create/Impulse` is your `Create` step applied to the synthetic source. `Create/FlatMap(<lambda at core.py:4094>)` is a `FlatMap` Beam inserted *for you* — `Create` is implemented as a `FlatMap` over a one-element impulse collection.

**Second, `beam.Create` silently adds a reshuffle.** Look at the `MaybeReshuffle/...` chain in the graph. This is Beam re-keying the data to break up the hot key that `Create` would otherwise produce. It is an optimisation, and it is also why `Create` is not the right tool for anything large — a `Reshuffle` on a real workload costs a full shuffle.

### Labels are your unit of debugging

When a Dataflow job fails, the error names the **label**, not the line of your Python file:

```text
Error in 'ParseOrders' step: ...
```

So a label is not documentation — it is an operational identifier. Two rules follow:

1. **Always label.** An unlabelled transform gets an auto-generated name like `ParDo(Split)`, which appears in every job graph and tells you nothing.
2. **Labels are unique within a pipeline.** Beam will not let you reuse one, and the error message does not tell you which two collided. We will reproduce that failure next.

## 7. A deliberate failure: duplicate transform label

Here is a real Beam bug you will hit eventually — usually when copy-pasting a stage and forgetting to rename it. We are going to trigger it on purpose so you recognise it later.

In [8]:
# This WILL fail. That is the point of the cell.
try:
    with beam.Pipeline() as p:
        _ = p | "ReadOrders" >> beam.Create([1, 2, 3])
        _ = p | "ReadOrders" >> beam.Map(lambda x: x)
except Exception as exc:
    print("Exception type:", type(exc).__name__)
    print()
    print("Message:")
    print(exc)


Exception type: RuntimeError

Message:
A transform with label "[9]: ReadOrders" already exists in the pipeline. To apply a transform with a specified label, write pvalue | "label" >> transform or use the option "auto_unique_labels" to automatically generate unique transform labels. Note "auto_unique_labels" could cause data loss when updating a pipeline or reloading the job state. This is not recommended for streaming jobs.


That is the verbatim Beam 2.77 message. Read it carefully:

- It says a transform with the label `"ReadOrders"` **already exists in the pipeline**.
- It does **not** say which two transforms collided, or on which line.
- It suggests `auto_unique_labels`, then immediately warns that this *"could cause data loss when updating a pipeline or reloading the job state"* — so it is **not** recommended for streaming jobs.

The static linter in this course (`resources/audit_course.py`) catches this class of mistake before you ever run anything, by parsing your source and checking for repeated `p | "Label" >>` within the same scope. That is the tool you want in CI; Beam's runtime error is the backstop.

### The fix

Rename the second label. Labels describe the *stage*, and two stages with the same name are two stages.

In [9]:
# The fix: give every stage its own label.
with beam.Pipeline() as p:
    (
        p
        | "Create" >> beam.Create([1, 2, 3])
        | "Double" >> beam.Map(lambda x: x * 2)
        | "Print" >> beam.Map(print)
    )
print()
print("Fixed — every label is unique.")


2
4
6

Fixed — every label is unique.


## 8. A real pipeline over real data

Now the full shape of a production pipeline: **read → transform → aggregate → write**. The course sample data is deterministic, so the numbers you see are the numbers the chapters quote.

The generator is seeded and idempotent, so we only run it when the data is missing.

In [10]:
import subprocess
import sys
from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent

if not (COURSE_ROOT / "sample_data" / "orders.csv").exists():
    print("Generating sample data...")
    subprocess.run(
        [sys.executable, str(COURSE_ROOT / "resources" / "sample_data.py"),
         "--output-dir", str(COURSE_ROOT / "sample_data")],
        check=True,
    )
else:
    print("Sample data already present — skipping generation.")

orders_csv = COURSE_ROOT / "sample_data" / "orders.csv"
print()
print("orders.csv:", orders_csv)
print("size:", f"{orders_csv.stat().st_size:,} bytes")
with orders_csv.open(encoding="utf-8") as fh:
    lines = fh.read().splitlines()
print("lines:", len(lines), "(including the header)")
print("header:", lines[0])
print("row 1 :", lines[1])


Sample data already present — skipping generation.

orders.csv: /Users/pvishnoi/PycharmProjects/data-etl-ml/data-engineering/data-Course/google/Dataflow/sample_data/orders.csv
size: 407,269 bytes
lines: 5001 (including the header)
header: order_id,customer_id,order_ts,region,channel,currency,status,amount,product_id
row 1 : o00000000,c000914,2026-01-15T00:54:38+00:00,AMER,web,GBP,complete,700.97,p0173


In [11]:
# The four-stage production shape.
import apache_beam as beam

INPUT = str(COURSE_ROOT / "sample_data" / "orders.csv")
OUTPUT = str(COURSE_ROOT / "out" / "nb01_first_pipeline")

with beam.Pipeline() as p:
    (
        p
        | "Read" >> beam.io.ReadFromText(INPUT)
        | "SkipHeader" >> beam.Filter(lambda line: not line.startswith("order_id"))
        | "Parse" >> beam.Map(lambda line: line.split(","))
        | "Valid" >> beam.Filter(lambda row: len(row) > 6 and row[6] == "complete")
        | "KeyByRegion" >> beam.Map(lambda row: (row[3], float(row[7])))
        | "SumRevenue" >> beam.CombinePerKey(sum)
        | "Format" >> beam.Map(lambda kv: f"{kv[0]},{kv[1]:.2f}")
        | "Write" >> beam.io.WriteToText(OUTPUT, num_shards=1, shard_name_template="")
    )

print()
print("Output files:")
for path in sorted((COURSE_ROOT / "out").glob("nb01_first_pipeline*")):
    print(" ", path.name)
    for line in path.read_text(encoding="utf-8").strip().splitlines():
        print("   ", line)



Output files:
  nb01_first_pipeline
    AMER,144082.57
    APAC,135554.57
    EMEA,150227.22
    LATAM,150197.64


### Two details worth stopping on

**First — the header.** `ReadFromText` reads every line, including `order_id,customer_id,...`. That line has no commas to split into nine fields, and `row[7]` would raise `IndexError` if it got that far. We skipped it explicitly with a `Filter`. Every CSV reader in the Beam ecosystem has this problem, and every one of them solves it by making you skip the first record yourself.

**Second — the output filename.** We passed `num_shards=1, shard_name_template=""`, which produces a file literally named `nb01_first_pipeline`. Without those options Beam writes `nb01_first_pipeline00000of00001` — a sharded file with a suffix. **Never hard-code the exact output filename in a script.** Use a glob, or force a single shard and an empty template as we did here. Chapter 3 covers this properly.

## 9. Common errors at this stage

| Error | Cause | Fix |
|---|---|---|
| `RuntimeError: A transform with label "X" already exists in the pipeline` | Two stages share a label | Rename one of them |
| `ValueError: Pipeline has validations errors: Missing GCS path option: temp_location` | Cloud runner without staging/temp | Pass `--staging_location` and `--temp_location` |
| `ModuleNotFoundError: No module named 'apache_beam'` | Wrong virtualenv | `which python`; `pip show apache-beam` |
| `FileNotFoundError: .../sample_data/orders.csv` | Sample data not generated | `python resources/sample_data.py --output-dir ./sample_data` |
| `IndexError: list index out of range` in a `Map` | The CSV header was not skipped | Add a `Filter` before the parse step |
| `PermissionDenied: 403 ... GCS` | Wrong service account | Grant `roles/storage.objectAdmin` on the bucket |
| `DefaultCredentialsError` | No ADC found | `gcloud auth application-default login` |
| Pipeline runs, no output file | No sink on the branch | Every branch must end in a sink |

---

## 🔑 Takeaways

- **Check the version first.** `beam.__version__` determines which runner behaviours you get; 2.77.0 is this course's baseline.
- The course runner is **`BundleBasedDirectRunner`**, not the Go-based `DirectRunner` — the Go runner panics on `AfterWatermark`, which the streaming chapter needs.
- **Local is the default.** `build_options([])` silently yields a local runner; credentials are opt-in, never required.
- **The cloud runner adds four requirements the local one never checks**: project, region, staging bucket, temp bucket. A green local run is not a green cloud run.
- **Prefer ADC and Workload Identity over key files.** Key files are long-lived secrets that leak from git history.
- **A PCollection is a description, not data.** Every branch must end in a sink or the pipeline does nothing.
- **Labels are your unit of debugging.** They must be unique per pipeline, and `resources/audit_course.py` catches repeats statically.
- **Never hard-code an output filename.** Beam suffixes shards; glob it or force `num_shards=1, shard_name_template=""`.

---

## 📚 Next

[02 · The Beam Model & Core Transforms](./02-beam-model-and-transforms.ipynb)